# scWAT Xenium slide QC summary

## Goal

Combine exactly four validated region bundles from one run label and execution mode. The summary compares two sections within each of `Mouse_1` and `Mouse_2` without treating cells as biological replicates and without inventing a left/right factor.


## Setup

HPC paths are defaults. Local validation injects D-drive paths and `LOCAL_SUBSET`. No package installation or writes outside `adipose_analysis` are permitted.


In [ ]:
EXECUTION_MODE <- toupper(Sys.getenv("SCWAT_QC_MODE", "FULL_HPC"))
PROJECT_ROOT <- Sys.getenv("SCWAT_PROJECT_ROOT", "/dssg/home/acct-svetoslav_chakarov/svetoslav_chakarov/Lab_members/Yanan_Hu/YNH_Xenium")
PIPELINE_REPO <- Sys.getenv("SCWAT_PIPELINE_REPO", file.path(PROJECT_ROOT, "adipose_analysis", "YNH_Xenium_scWAT"))
RUN_LABEL <- Sys.getenv("SCWAT_RUN_LABEL", "scwat_qc_hpc")
RUN_ROOT <- Sys.getenv("SCWAT_RUN_ROOT", file.path(PROJECT_ROOT, "adipose_analysis", "scwat_qc_outputs", RUN_LABEL))
TEMP_ROOT <- Sys.getenv("SCWAT_TEMP_ROOT", file.path(PROJECT_ROOT, "adipose_analysis", "tmp", RUN_LABEL))
dir.create(TEMP_ROOT, recursive = TRUE, showWarnings = FALSE)
Sys.setenv(TMPDIR = TEMP_ROOT, TMP = TEMP_ROOT, TEMP = TEMP_ROOT)
source(file.path(PIPELINE_REPO, "R", "source.R"))
regions <- expected_scwat_regions()


## Inputs and integrity

Require Region_1 through Region_4, reject missing or duplicate regions and mixed run labels or modes, then reload every saved sparse object and table.


In [ ]:
manifest <- utils::read.delim(file.path(PIPELINE_REPO, "config", "scwat_sample_manifest.tsv"), check.names = FALSE)
manifest_check <- validate_sample_manifest(manifest, regions)
stopifnot(isTRUE(manifest_check$valid))
slide_data <- read_scwat_slide_qc_outputs(RUN_ROOT, expected_regions = regions, run_label = RUN_LABEL, execution_mode = EXECUTION_MODE)
stopifnot(identical(slide_data$coverage$region_id, regions))


## Cell QC

Summarize fixed-bound pass fractions and review flags. `primary_include` is never redefined at slide level. Region readiness is derived from current evidence rather than inherited anchor or sensitivity labels.


In [ ]:
slide_summary <- summarise_slide_qc(slide_data)
mouse_summary <- summarise_scwat_mouse_sections(slide_summary$section_summary, manifest)
mouse_summary$within_mouse
mouse_summary$mouse_summary


## Outputs and checks

Save combined tables, descriptive within-mouse and mouse-level summaries, readiness gates, figures, session information, and a reloadable slide object.


In [ ]:
summary_output_dir <- file.path(RUN_ROOT, "slide_summary")
result <- write_scwat_slide_qc_bundle(PROJECT_ROOT, summary_output_dir, RUN_LABEL, EXECUTION_MODE, slide_data, slide_summary, mouse_summary)
stopifnot(validate_scwat_slide_qc_bundle(summary_output_dir, expected_regions = regions))
result


## Next steps

Use full-HPC evidence for the final initial-QC decision. With two mice, mouse and section comparisons are descriptive. Only after these gates pass should Region 3 be built as the anchor, Regions 1-2 be tested for admission, and Region 4 be handled as mapping-only sensitivity data.
